# Store Sales Preprocessing

Seven competition tables arrive at different grains. This notebook resolves
them into two stable interfaces used by the rest of the system, with each
feature traceable to its source.

**Required input:** seven untouched CSV files in `data/raw/`.

**Outputs:**

- `data/processed/00_STORE_SALES_EDA.csv`: labeled history for EDA and model development;
- `data/processed/01_STORE_SALES_KAGGLE_TEST.csv`: future rows without `sales` for Kaggle inference.

## Workflow

| Stage | Operation | Result |
|---|---|---|
| Source contract | Confirm files, grains, and join keys | Accepted raw tables |
| Raw quality | Profile schemas, keys, missing values, and calendar gaps | Documented source conditions |
| Enrichment | Add store, calendar, oil, transaction, and scheduled-event context | Two in-memory feature tables |
| Validation | Reconcile rows, IDs, keys, targets, and allowed history gaps | Accepted processed interfaces |
| Persistence | Replace the two processed CSVs and verify their headers | Inputs for notebooks 02 and 03 |

Model-specific encoding, imputation, and sales lags are added later during
training.


In [ ]:
from pathlib import Path
import pandas as pd
from IPython.display import display

RAW_DIR = Path("data/raw")
PROCESSED_DIR = Path("data/processed")
FINAL_TRAIN_PATH = PROCESSED_DIR / "00_STORE_SALES_EDA.csv"
FINAL_KAGGLE_TEST_PATH = PROCESSED_DIR / "01_STORE_SALES_KAGGLE_TEST.csv"

if not Path("store_sales_preprocessing.py").is_file() or not RAW_DIR.is_dir():
    raise FileNotFoundError(
        "Open this notebook from the AI Engineer project root. The preprocessing "
        "module and private raw-data directory must already exist."
    )

from store_sales_preprocessing import (
    ALLOWED_MISSING_COLUMNS,
    BASE_KEY,
    CALENDAR_COLUMNS,
    HOLIDAY_FEATURE_COLUMNS,
    OIL_FEATURE_COLUMNS,
    OIL_NULLABLE_COLUMNS,
    STORE_OUTPUT_COLUMNS,
    TEST_COLUMNS,
    TRAIN_COLUMNS,
    TRANSACTION_FEATURE_COLUMNS,
    TRANSACTION_NULLABLE_COLUMNS,
    add_calendar_features,
    attach_many_to_one,
    build_oil_lookup,
    build_transaction_lookup,
    join_store_metadata,
    normalize_holidays,
    read_base_table,
    read_holidays,
    read_oil,
    read_store_table,
    read_transactions,
    validate_final_table,
    write_plain_csv,
)

EXPECTED_TRAIN_COLUMNS = (
    TRAIN_COLUMNS
    + STORE_OUTPUT_COLUMNS
    + CALENDAR_COLUMNS
    + OIL_FEATURE_COLUMNS
    + TRANSACTION_FEATURE_COLUMNS
    + HOLIDAY_FEATURE_COLUMNS
)
EXPECTED_TEST_COLUMNS = (
    TEST_COLUMNS
    + STORE_OUTPUT_COLUMNS
    + CALENDAR_COLUMNS
    + OIL_FEATURE_COLUMNS
    + TRANSACTION_FEATURE_COLUMNS
    + HOLIDAY_FEATURE_COLUMNS
)

print("Preprocessing environment configured successfully.")


## 1. Validate the source contract

Start with the seven required files and their expected keys. `source_contract`
lists each table's grain, forecasting role, availability, and local size.


In [ ]:
source_contract = pd.DataFrame(
    [
        ["train.csv", "date x store x family", "Historical sales target and promotions", "date + store_nbr + family"],
        ["test.csv", "date x store x family", "Future rows that need predictions", "date + store_nbr + family"],
        ["stores.csv", "store", "City, state, store type, and cluster", "store_nbr"],
        ["oil.csv", "date", "Daily oil-price history", "date"],
        ["transactions.csv", "date x store", "Historical store transaction counts", "date + store_nbr"],
        ["holidays_events.csv", "event", "National, regional, and local calendar events", "not unique by date"],
        ["sample_submission.csv", "future prediction row", "Required Kaggle submission format", "id"],
    ],
    columns=["file", "source_grain", "role", "expected_key"],
)

source_contract["available"] = source_contract["file"].map(
    lambda name: (RAW_DIR / name).is_file()
)
source_contract["size_mb"] = source_contract["file"].map(
    lambda name: (RAW_DIR / name).stat().st_size / 1024**2
    if (RAW_DIR / name).is_file()
    else None
)

display(source_contract)

assert source_contract["available"].all(), "One or more contracted raw files are missing."

## 2. Load and profile the raw tables

Load the six preprocessing inputs with explicit schemas. `raw_profiles` captures
shape, missing cells, duplicate keys, and date coverage;
`sample_submission.csv` is reserved for the final ordering check.


In [ ]:
raw_train = read_base_table(RAW_DIR / "train.csv", TRAIN_COLUMNS)
raw_kaggle_test = read_base_table(RAW_DIR / "test.csv", TEST_COLUMNS)
stores = read_store_table(RAW_DIR / "stores.csv")
oil = read_oil(RAW_DIR / "oil.csv")
transactions = read_transactions(RAW_DIR / "transactions.csv")
holidays = read_holidays(RAW_DIR / "holidays_events.csv")


def profile_table(
    name: str,
    table: pd.DataFrame,
    key: list[str],
) -> dict[str, object]:
    """Return public-safe structure and quality information for one table."""
    return {
        "table": name,
        "rows": len(table),
        "columns": table.shape[1],
        "missing_cells": int(table.isna().sum().sum()),
        "duplicate_key_rows": int(table.duplicated(key, keep=False).sum()),
        "date_min": (
            table["date"].min().date().isoformat() if "date" in table else None
        ),
        "date_max": (
            table["date"].max().date().isoformat() if "date" in table else None
        ),
    }


raw_profiles = pd.DataFrame(
    [
        profile_table("train.csv", raw_train, BASE_KEY),
        profile_table("test.csv", raw_kaggle_test, BASE_KEY),
        profile_table("stores.csv", stores, ["store_nbr"]),
        profile_table("oil.csv", oil, ["date"]),
        profile_table("transactions.csv", transactions, ["date", "store_nbr"]),
        profile_table("holidays_events.csv", holidays, ["date"]),
    ]
)

display(raw_profiles)

## 3. Classify the raw-data gaps

Missing records need different rules. Oil can use an older published quote; an
absent transaction row cannot be treated as zero activity. Zero sales remain
valid targets, while the four absent Christmas dates stay unobserved.

`raw_conditions` separates planned events, the Manabi earthquake sequence, and
event descriptions that need an availability rule. Source values and rows stay
unchanged.


In [ ]:
complete_calendar = pd.date_range(
    raw_train["date"].min(), raw_train["date"].max(), freq="D"
)
missing_calendar_dates = complete_calendar.difference(raw_train["date"].unique())

event_mask = holidays["type"].eq("Event")
event_description = holidays["description"].str.lower()
planned_event_mask = event_mask & (
    event_description.str.contains("dia de la madre", regex=False)
    | event_description.str.contains("mundial de futbol", regex=False)
    | event_description.str.contains("black friday", regex=False)
    | event_description.str.contains("cyber monday", regex=False)
)
earthquake_event_mask = event_mask & event_description.str.contains(
    "terremoto manabi", regex=False
)
unknown_event_mask = event_mask & ~planned_event_mask & ~earthquake_event_mask

raw_conditions = pd.Series(
    {
        "train_zero_sales_rows": int(raw_train["sales"].eq(0).sum()),
        "train_zero_sales_rate_pct": 100 * raw_train["sales"].eq(0).mean(),
        "train_fractional_sales_rows": int(raw_train["sales"].mod(1).ne(0).sum()),
        "train_promoted_rows": int(raw_train["onpromotion"].gt(0).sum()),
        "missing_oil_prices": int(oil["dcoilwtico"].isna().sum()),
        "planned_event_rows": int(planned_event_mask.sum()),
        "earthquake_event_rows_excluded_later": int(earthquake_event_mask.sum()),
        "unknown_event_rows": int(unknown_event_mask.sum()),
        "missing_calendar_date_count": len(missing_calendar_dates),
        "missing_calendar_dates": ", ".join(
            date.date().isoformat() for date in missing_calendar_dates
        ),
        "holiday_rows_on_non_unique_dates": int(
            holidays.duplicated(["date"], keep=False).sum()
        ),
    },
    name="value",
).to_frame()

display(raw_conditions)
assert int(unknown_event_mask.sum()) == 0

## 4. Attach store metadata

Join `city`, `state`, `store_type`, and `store_cluster` by `store_nbr`.
`store_join_summary` checks row counts, ID order, business keys, and metadata
coverage after the many-to-one join.


In [ ]:
train_with_stores, train_store_evidence = join_store_metadata(
    "train.csv", raw_train, stores
)
kaggle_test_with_stores, test_store_evidence = join_store_metadata(
    "test.csv", raw_kaggle_test, stores
)

store_join_summary = pd.DataFrame(
    [
        {"table": "train", **train_store_evidence},
        {"table": "kaggle_test", **test_store_evidence},
    ]
)

display(store_join_summary)
print("Added columns: city, state, store_type, store_cluster")

## 5. Derive the accepted calendar features

Keep raw `date` for joins, lags, splits, and forecast output. The model receives
`month`, `day_of_month`, and `day_of_week`, with Monday=1 and Sunday=7.
`calendar_summary` checks coverage in both tables.


In [ ]:
train_stage = add_calendar_features(train_with_stores)
kaggle_test_stage = add_calendar_features(kaggle_test_with_stores)

calendar_meanings = {
    "month": "1=January, 12=December",
    "day_of_month": "calendar date from 1 to 31",
    "day_of_week": "1=Monday, 7=Sunday",
}

calendar_summary = pd.DataFrame(
    {
        "feature": CALENDAR_COLUMNS,
        "meaning": [calendar_meanings[column] for column in CALENDAR_COLUMNS],
        "minimum_train_value": [
            int(train_stage[column].min()) for column in CALENDAR_COLUMNS
        ],
        "maximum_train_value": [
            int(train_stage[column].max()) for column in CALENDAR_COLUMNS
        ],
        "known_before_forecast": True,
        "missing_cells_train": [
            int(train_stage[column].isna().sum()) for column in CALENDAR_COLUMNS
        ],
    }
)

display(calendar_summary)


## 6. Align oil history to the forecast cutoff

Oil is aligned to the forecast cutoff at 16, 21, 28, and 35 days. If a reference
date has no quote, use the latest quote already published and record its age.

`oil_evidence` summarizes the four lag values and their source ages.


In [ ]:
all_dates = pd.concat(
    [train_stage["date"], kaggle_test_stage["date"]],
    ignore_index=True,
)

oil_lookup, oil_evidence = build_oil_lookup(all_dates, oil)
train_stage = attach_many_to_one(
    "train oil join",
    train_stage,
    oil_lookup,
    ["date"],
    OIL_FEATURE_COLUMNS,
    OIL_NULLABLE_COLUMNS,
)
kaggle_test_stage = attach_many_to_one(
    "test oil join",
    kaggle_test_stage,
    oil_lookup,
    ["date"],
    OIL_FEATURE_COLUMNS,
    OIL_NULLABLE_COLUMNS,
)

display(pd.Series(oil_evidence, name="oil_evidence").to_frame())

## 7. Align transaction history to the forecast cutoff

Transactions use exact store-level lags and are not carried forward. Missing
source records stay missing; `transactions_lag_available_count` records how many
of the four references are available.


In [ ]:
all_store_dates = pd.concat(
    [
        train_stage[["date", "store_nbr"]],
        kaggle_test_stage[["date", "store_nbr"]],
    ],
    ignore_index=True,
)

transaction_lookup, transaction_evidence = build_transaction_lookup(
    all_store_dates,
    transactions,
)
train_stage = attach_many_to_one(
    "train transaction join",
    train_stage,
    transaction_lookup,
    ["date", "store_nbr"],
    TRANSACTION_FEATURE_COLUMNS,
    TRANSACTION_NULLABLE_COLUMNS,
)
kaggle_test_stage = attach_many_to_one(
    "test transaction join",
    kaggle_test_stage,
    transaction_lookup,
    ["date", "store_nbr"],
    TRANSACTION_FEATURE_COLUMNS,
    TRANSACTION_NULLABLE_COLUMNS,
)

display(pd.Series(transaction_evidence, name="transaction_evidence").to_frame())

## 8. Map forecast-known calendar records to stores

National records apply to every store, regional records to the matching state,
and local records to the matching city. Active holidays, special working days,
transfer sources, transfer destinations, and planned events remain distinct
operating signals. Mother's Day, World Cup, Black Friday, and Cyber Monday share
one planned-event flag.

The Manabi earthquake sequence is excluded because it was not knowable before
the event. `holiday_evidence` verifies the eight store-aware calendar fields and
stops the workflow if a new event description has no explicit rule.


In [ ]:
holiday_lookup, holiday_evidence = normalize_holidays(holidays, stores)

train_stage = train_stage.merge(
    holiday_lookup,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one",
    sort=False,
)
kaggle_test_stage = kaggle_test_stage.merge(
    holiday_lookup,
    on=["date", "store_nbr"],
    how="left",
    validate="many_to_one",
    sort=False,
)

for table in [train_stage, kaggle_test_stage]:
    table[HOLIDAY_FEATURE_COLUMNS] = (
        table[HOLIDAY_FEATURE_COLUMNS].fillna(0).astype("int16")
    )

display(pd.Series(holiday_evidence, name="holiday_evidence").to_frame())

## 9. Validate row and target preservation

Reconcile the enriched tables with the original prediction populations.
`final_validation` checks row counts, ID order, business keys, target
preservation, and the accepted missing-value boundary.


In [ ]:
train_validation = validate_final_table(
    "train",
    raw_train,
    train_stage,
    has_target=True,
)
test_validation = validate_final_table(
    "competition test",
    raw_kaggle_test,
    kaggle_test_stage,
    has_target=False,
)

final_validation = pd.DataFrame(
    [
        {"table": "data/processed/00_STORE_SALES_EDA.csv", **train_validation},
        {"table": "data/processed/01_STORE_SALES_KAGGLE_TEST.csv", **test_validation},
    ]
)

display(final_validation)

## 10. Reconcile the before-and-after structure

`before_after` compares the raw and enriched shapes; `train_added_columns` lists
the full transformation footprint.


In [ ]:
train_added_columns = [
    column for column in train_stage.columns if column not in raw_train.columns
]
test_added_columns = [
    column
    for column in kaggle_test_stage.columns
    if column not in raw_kaggle_test.columns
]

def summarize_structure(name, raw, final, added_columns):
    missing = final.isna().sum()
    unexpected_missing = missing.loc[
        (missing > 0) & (~missing.index.isin(ALLOWED_MISSING_COLUMNS))
    ]
    return {
        "table": name,
        "raw_rows": len(raw),
        "final_rows": len(final),
        "raw_columns": raw.shape[1],
        "final_columns": final.shape[1],
        "added_columns": len(added_columns),
        "expected_history_gap_cells": int(missing.sum()),
        "unexpected_missing_cells": int(unexpected_missing.sum()),
        "final_duplicate_base_keys": int(
            final.duplicated(BASE_KEY, keep=False).sum()
        ),
    }

before_after = pd.DataFrame(
    [
        summarize_structure("labeled history", raw_train, train_stage, train_added_columns),
        summarize_structure(
            "Kaggle inference",
            raw_kaggle_test,
            kaggle_test_stage,
            test_added_columns,
        ),
    ]
)

display(before_after)
display(pd.DataFrame({"added_train_column": train_added_columns}))
assert before_after["unexpected_missing_cells"].eq(0).all()

## 11. Save and verify the processed interfaces

Write the two processed CSVs and read their headers back against the canonical
EDA and modeling contracts. `interface_check` records the final handoff.


In [ ]:
assert train_stage.columns.tolist() == EXPECTED_TRAIN_COLUMNS
assert kaggle_test_stage.columns.tolist() == EXPECTED_TEST_COLUMNS

write_plain_csv(train_stage, FINAL_TRAIN_PATH)
write_plain_csv(kaggle_test_stage, FINAL_KAGGLE_TEST_PATH)

saved_train_columns = pd.read_csv(FINAL_TRAIN_PATH, nrows=0).columns.tolist()
saved_test_columns = pd.read_csv(FINAL_KAGGLE_TEST_PATH, nrows=0).columns.tolist()

interface_check = pd.Series(
    {
        "train_columns_match_contract": saved_train_columns == EXPECTED_TRAIN_COLUMNS,
        "kaggle_columns_match_contract": saved_test_columns == EXPECTED_TEST_COLUMNS,
        "train_target_preserved": train_stage["sales"].equals(raw_train["sales"]),
        "kaggle_test_has_no_sales_target": "sales" not in saved_test_columns,
        "train_output_exists": FINAL_TRAIN_PATH.is_file(),
        "kaggle_output_exists": FINAL_KAGGLE_TEST_PATH.is_file(),
    },
    name="passed",
).to_frame()

display(interface_check)
assert interface_check["passed"].all()

print(f"Saved labeled data : {FINAL_TRAIN_PATH}")
print(f"Saved Kaggle data  : {FINAL_KAGGLE_TEST_PATH}")
print("PREPROCESSING COMPLETE: the processed interfaces match the final contract.")


## Result

The run preserves source rows, IDs, business keys, and labeled targets while
adding store, calendar, oil, transaction, and planned-event context. Genuine
historical gaps are carried forward for train-fitted imputation and missing
indicators.

The two validated CSV interfaces feed the EDA and modeling notebooks.
